## Agentic RAG

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
BASE_URL = os.getenv('OPENAI_BASE_URL')

os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ["LANGCHAIN_TRACING_V2"] = "true" 
os.environ['LANGSMITH_PROJECT'] = 'AgenticAITraining' 

### Indexing

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma

In [6]:
# Constants
MODEL = 'gpt-5.1'
EMBD_MODEL = 'text-embedding-3-small'
FILE_PATH = '../data/llm_guide.pdf'
PERSIST_DIRECTORY = "./simple-rag-db"
VECTOR_STORE_NAME = "simple-rag"

In [42]:
llm = ChatOpenAI(
    api_key = API_KEY, 
    base_url = BASE_URL, 
    model = MODEL
).with_config(
    run_name=f"{MODEL}"
)

embedding_model = OpenAIEmbeddings(
    api_key = API_KEY,
    base_url = BASE_URL,
    model = EMBD_MODEL
)

In [43]:
loader = PyPDFLoader(FILE_PATH)
documents = loader.load()

In [44]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500, 
    chunk_overlap = 50
)

chunks = text_splitter.split_documents(documents)

In [45]:
if os.path.exists(PERSIST_DIRECTORY):
    vector_store = Chroma(
        collection_name = VECTOR_STORE_NAME,
        embedding_function = embedding_model,
        persist_directory = PERSIST_DIRECTORY
    )

    existing_count = len(vector_store.get())
    new_count = len(chunks)

    if existing_count != new_count:
        vector_store.delete_collection()
        vector_store = Chroma(
            collection_name = VECTOR_STORE_NAME,
            embedding_function = embedding_model,
            persist_directory = PERSIST_DIRECTORY
        )
        vector_store.add_documents(chunks)
else:
    vector_store = Chroma(
        collection_name = VECTOR_STORE_NAME,
        embedding_function = embedding_model,
        persist_directory = PERSIST_DIRECTORY
    )
    vector_store.add_documents(chunks)

### Reterieval and Generateion

In [51]:
from langchain.tools import tool

@tool("PDFRetriever", response_format = 'content_and_artifact')
def retrieve_context(query: str, num_docs: int = 3):
    '''Reterive information to help answer the query form the database.'''
    reterieved_docs = vector_store.similarity_search(query, k = num_docs)
    serialized = "\n\n".join((f'Source: {doc.metadata['source']}\nContent: {doc.page_content}') for doc in reterieved_docs)
    return serialized, reterieved_docs

In [52]:
query = 'How did LLM evolved after introduction of ChatGPT'

### Creating Agent for reterival

In [53]:
from langchain.agents import create_agent

tools = [retrieve_context]

prompt = (
    """You have access to a tool that retrieves context from a pdf.
    Use the tool to help answer user queries.
    Answer in short, only using the Context provided"""
)

agent = create_agent(llm, tools, system_prompt=prompt)

In [54]:
for event in agent.stream(
    {'messages': [{'role': 'user', 'content': query}]},
    stream_mode='values'
):
    event['messages'][-1].pretty_print()

================================ Human Message =================================

How did LLM evolved after introduction of ChatGPT
================================== Ai Message ==================================
Tool Calls:
  PDFRetriever (call_Ucy8xdypPktlZk0Cj115Q7D2)
 Call ID: call_Ucy8xdypPktlZk0Cj115Q7D2
  Args:
    query: evolution of LLMs after ChatGPT introduction
    num_docs: 3
================================= Tool Message =================================
Name: PDFRetriever

Source: ../data/llm_guide.pdf
Content: explosive charge that brought LLMs into the mainstream. ChatGPT provides 
a nice user interface (or API) where users can feed prompts to one of many 
models (GPT-3.5, GPT-4, and more) and typically get a fast response. These are 
among the highest-performing models, trained on enormous data sets, and are 
capable of extremely complex tasks both from a technical standpoint, such as 
code generation, as well as from a creative perspective like writing poetry in a 
s